# Track 1 automatic crop ranking
Compare released object semantics against automatically segmented crops; no manual boxes, masks, poses or geometry labels.

In [ ]:
import sys,subprocess,json,hashlib,urllib.request
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','transformers==5.3.0'],check=True)
Path('/tmp/select_seed.py').write_text('"""Select automatic video seeds using grounded proposals and semantic margins."""\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom transformers import AutoProcessor,AutoModel,AutoModelForZeroShotObjectDetection\n\nDESCRIPTIONS={\n \'hula_hoop\':[\'a thin circular hula hoop\',\'a plastic exercise hoop\'],\n \'big_red_bowl\':[\'a red plastic bowl\',\'a large red mixing bowl\'],\n \'iron\':[\'a blue electric clothes iron\',\'a metallic steam iron with blue accents\'],\n \'white_desk\':[\'a white rectangular office desk with two legs\',\'a white standing desk\'],\n \'black_pan\':[\'a black frying pan with a handle\',\'a black cooking pan\'],\n \'foam_grass_block\':[\'a green and brown pixelated Minecraft toy cube\',\'a foam cube with green and brown squares\'],\n \'paint_roller\':[\'a white paint roller with a blue handle\',\'a paint roller tool with a handle\'],\n \'pink_foam_roll\':[\'a pink cylindrical foam exercise roller\',\'a pink foam cylinder with a hole\'],\n \'short_wood_stool\':[\'a short wooden four-legged stool\',\'a wooden square stool\'],\n \'white_laptop_cart\':[\'a white rolling laptop cart with a central column and wheels\',\'a white mobile laptop stand with casters\'],\n}\nQUERIES={\n \'iron\':[\'a clothes iron.\',\'a blue electric iron.\'],\n \'foam_grass_block\':[\'a green brown cube.\',\'a Minecraft block.\',\'a foam cube.\'],\n \'white_desk\':[\'a white desk.\',\'an office desk.\'],\n \'white_laptop_cart\':[\'a rolling laptop stand.\',\'a white cart with wheels.\'],\n \'paint_roller\':[\'a paint roller.\',\'a paint roller with handle.\'],\n}\nDISTRACTORS=[\'a camera tripod\',\'an office chair\',\'a curtain\',\'a person\',\'a slipper\',\n             \'a long metal rail\',\'a wall\',\'a wooden floor\',\'a table lamp\',\'an ironing board\']\n\n\ndef select_seed(frames:Path,frame_count:int,height:int,width:int,prompt:str,object_class:str):\n    if object_class not in DESCRIPTIONS:raise ValueError(\'Unknown released object class\')\n    detector_id=\'IDEA-Research/grounding-dino-tiny\'\n    processor=AutoProcessor.from_pretrained(detector_id)\n    detector=AutoModelForZeroShotObjectDetection.from_pretrained(detector_id).to(\'cuda\').eval()\n    candidates=[];crops=[]\n    queries=list(dict.fromkeys(QUERIES.get(object_class,[DESCRIPTIONS[object_class][0]+\'.\'])+[prompt]))\n    sampled=sorted(set(np.linspace(0,frame_count-1,12).astype(int).tolist()+[min(frame_count-1,x) for x in (0,15,30,60,120)]))\n    for frame in sampled:\n        image=Image.open(frames/f\'{frame:06d}.jpg\').convert(\'RGB\')\n        def detect(text):\n            inputs=processor(images=image,text=text,return_tensors=\'pt\').to(\'cuda\')\n            with torch.inference_mode():output=detector(**inputs)\n            return processor.post_process_grounded_object_detection(output,inputs.input_ids,\n                threshold=.12,text_threshold=.12,target_sizes=[(height,width)])[0]\n        people=detect(\'a person.\')\n        if not len(people[\'boxes\']):continue\n        human_id=int(people[\'scores\'].argmax())\n        human={\'box\':people[\'boxes\'][human_id].cpu().tolist(),\'confidence\':float(people[\'scores\'][human_id])}\n        seen=[]\n        for query in queries:\n            result=detect(query)\n            for box,confidence in zip(result[\'boxes\'].cpu().numpy(),result[\'scores\'].cpu().numpy()):\n                x0,y0,x1,y1=box\n                if x1-x0<8 or y1-y0<8:continue\n                if any(np.max(np.abs(box-prev))<3 for prev in seen):continue\n                seen.append(box)\n                margin=.06*max(x1-x0,y1-y0)\n                crop=image.crop((max(0,int(x0-margin)),max(0,int(y0-margin)),min(width,int(x1+margin+1)),min(height,int(y1+margin+1))))\n                candidates.append({\'frame\':frame,\'human\':human,\'object\':{\'box\':box.astype(float).tolist(),\'confidence\':float(confidence)},\'query\':query})\n                crops.append(crop)\n    del processor,detector;torch.cuda.empty_cache()\n    if not candidates:raise ValueError(\'No automatic grounded candidates\')\n    model_id=\'google/siglip2-base-patch16-224\'\n    processor=AutoProcessor.from_pretrained(model_id)\n    model=AutoModel.from_pretrained(model_id).to(\'cuda\').eval()\n    target_text=DESCRIPTIONS[object_class]\n    other_text=[text for key,values in DESCRIPTIONS.items() if key!=object_class for text in values]+DISTRACTORS\n    texts=[\'This is a photo of \'+text+\'.\' for text in target_text+other_text]\n    for start in range(0,len(crops),16):\n        inputs=processor(text=texts,images=crops[start:start+16],padding=\'max_length\',max_length=64,truncation=True,return_tensors=\'pt\').to(\'cuda\')\n        with torch.inference_mode(),torch.autocast(\'cuda\',dtype=torch.float16):logits=model(**inputs).logits_per_image.float().cpu().numpy()\n        if not np.isfinite(logits).all():raise ValueError(\'Nonfinite semantic ranking logits\')\n        for offset,row in enumerate(logits):\n            item=candidates[start+offset]\n            positive=float(row[:len(target_text)].max());negative=float(row[len(target_text):].max())\n            item.update(semantic_margin=positive-negative,semantic_target_logit=positive,\n                        semantic_best_label=(target_text+other_text)[int(row.argmax())],semantic_model=model_id)\n            item[\'score\']=positive-negative+.1*positive+.25*item[\'object\'][\'confidence\']\n    eligible=[c for c in candidates if c[\'semantic_margin\']>0]\n    if not eligible:raise ValueError(\'No candidate beats the released alternative classes/distractors\')\n    seed=max(eligible,key=lambda c:c[\'score\'])\n    del model,processor;torch.cuda.empty_cache()\n    return seed,candidates,detector_id\n')
sys.path.insert(0,'/tmp')
from select_seed import DESCRIPTIONS,DISTRACTORS
from transformers import AutoModel,AutoProcessor
from PIL import Image
import torch,numpy as np
torch.set_num_threads(2)
model_id='google/siglip2-base-patch16-224'
processor=AutoProcessor.from_pretrained(model_id)
model=AutoModel.from_pretrained(model_id).eval()
texts=[];labels=[]
for name,descriptions in DESCRIPTIONS.items():
 for text in descriptions:labels.append(name);texts.append('This is a photo of '+text+'.')
for text in DISTRACTORS:labels.append('distractor:'+text);texts.append('This is a photo of '+text+'.')
url='https://huggingface.co/datasets/nvidia/video_to_data_challenge/resolve/main/track_1/meta/episodes_metadata.jsonl'
with urllib.request.urlopen(url) as f:metadata={x['episode_index']:x for x in map(json.loads,f.read().decode().splitlines())}
all_paths=sorted(Path('/kaggle/input').rglob('object_observation.png'))
chosen={}
for path in all_paths:
 episode=int(path.parent.name.split('_')[-1])
 semantic=json.loads(path.with_name('segmentation_report.json').read_text()).get('semantic_selection',False)
 if episode not in chosen or semantic:chosen[episode]=path
paths=[chosen[key] for key in sorted(chosen)]
assert len(paths)==30,len(paths)
records=[]
for start in range(0,len(paths),4):
 batch=paths[start:start+4]
 inputs=processor(text=texts,images=[Image.open(p).convert('RGB') for p in batch],padding='max_length',max_length=64,truncation=True,return_tensors='pt')
 with torch.inference_mode():logits=model(**inputs).logits_per_image.float().numpy()
 for path,row in zip(batch,logits):
  episode=int(path.parent.name.split('_')[-1]);name=metadata[episode]['object']
  own=[i for i,label in enumerate(labels) if label==name];other=[i for i,label in enumerate(labels) if label!=name]
  record={'episode':episode,'object':name,'input_sha256':hashlib.sha256(path.read_bytes()).hexdigest(),
   'target_logit':float(row[own].max()),'semantic_margin':float(row[own].max()-row[other].max()),
   'predicted_class':labels[int(row.argmax())],'model':model_id,'model_revision':getattr(model.config,'_commit_hash',None)}
  records.append(record);print('CROP_RESULT',json.dumps(record),flush=True)
Path('/kaggle/working/semantic_crop_ranking.json').write_text(json.dumps(records,indent=2))
